## About
- This notebook includes the exploratory analysis of the ALD Olink liver inflammation biomarker study. Specifically, it includes
1. PCA of all samples in the discvoery set based on plasma proteome
2. Global correlation between plasma proteome and clinical variables
3. Correlation of plasma proteome to liver inflammation score measued by the sum of lobular inflammation and hepatocyte ballooning as defined in the NAFLD activity score.

### Input
1. curated proteomics and clinical data
2. shortlisted and novel cytokines 

### Output
1. PCA
2. global correlation plot
3. Correlation summary stats

In [ ]:
# Load packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.decomposition import PCA
import pingouin as pg
import statsmodels.stats.multitest as multi

### Import curated proteomics and clinical data

In [ ]:
# Define project directory
Base = 'data_directory'

df_olink = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='proteomics_curated').set_index('CBMRID')
df_cli = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='clinical_curated').set_index('CBMRID')

In [ ]:
olink_proteins = df_olink.columns
addon = ['alt_log2', 'ast_log2']

In [ ]:
# Join curated proteomics and clinical data
data = df_olink.join(df_cli)
# Drop samples without outcome target measure
data = data[data['cohort']!='GALA_LiHep'].dropna(subset=['inflam_binary'])

In [ ]:
data['cohort'].value_counts()

In [ ]:
data.shape

### Correlation to inflammatory activity

In [ ]:
# Rows whose histology-based inflammation score is na, but nonetheless assigned mild inflammatory activity based on defined criteria
mask = data['inflammation'].isna() & data['inflam_binary'].notnull()
data.loc[mask,'inflammation']=data.loc[mask]['inflam_binary']

In [ ]:
# Slice discovery cohorts
#data_validation = data[data['cohort']=='SIP']
discovery_cohorts = ['ALD', 'HP', 'RDC']
data=data[data['cohort'].isin(discovery_cohorts)]

In [ ]:
# import MS data 
data_ms = pd.read_csv(os.path.join(Base, 'data_curated/olink_ms_combined.csv')).set_index('CBMRID')
data=data.join(data_ms[['Q08380']])

In [ ]:
olink_ms_proteins = olink_proteins.tolist() + ['Q08380']
olink_ms_proteins_altast = olink_ms_proteins+addon
olink_altast = olink_proteins.tolist() + addon

#### PCA

In [ ]:
# PCA of all individuals in the discovery cohorts (GALA-ALD/HP, REDUCTION)
from sklearn.decomposition import PCA
pca = PCA(10)
df = data[olink_ms_proteins + ['cohort', 'inflam_binary', ]]
X = df[olink_proteins]
pca.fit(X)
variance_1 = pca.explained_variance_[0].round(1)
variance_2 = pca.explained_variance_[1].round(1)
X = pca.transform(X)
#c = df['cohort'].map({'ALD':'#3F3E9E', 'HP':'#DDD6C5', 'RDC':'#007A87'})
c = df['inflam_binary'].map({1:'steelblue', 0:'lightgray'})

fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x=X[:, 0], y=X[:, 1], c=c, s=10)
plt.xlabel('PC1 ({}%)'.format(variance_1), fontsize=12)
plt.ylabel('PC2 ({}%)'.format(variance_2), fontsize=12)
plt.legend(['I2-5', 'I0-1'])
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(Base, 'exploratory_analysis/pca.png'), dpi=120, bbox_inches='tight')

#### Global correlation

In [ ]:
g=sns.clustermap(data[olink_altast + ['inflammation']].corr(), 
               cmap ='bwr', yticklabels=True, xticklabels=True)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize = 7);
plt.rcParams['pdf.fonttype'] = 42
g.savefig(os.path.join(Base, 'exploratory_analysis/heatmap.pdf'), dpi=120, bbox_inches='tight')

In [ ]:
shortlist = pd.read_excel(os.path.join(Base, 'Shortlisting_of_cytokines.xlsx'))
shortlisted = shortlist['Protein'].tolist()
novel=shortlist[shortlist['novel']==1]['Protein']
# Remove proteins that are known to be no longer novel
novel_nomore = ['CDCP1']
novel = list(set(novel) - set(novel_nomore))

In [ ]:
# ("TWEAK" AND ("liver inflammation" OR "hepatic inflammation" OR "NASH" OR "steatohepatitis)) AND (NASH OR liver inflammation)

### Compute partial correlation

In [ ]:
covar = ['age', 'bmi', 'gender']
corr = pg.pairwise_corr(data=data, columns=[['inflammation'], olink_ms_proteins_altast], covar=covar, method='spearman').set_index('Y')
corr['abs(r)']=abs(corr['r'])
corr=corr.sort_values(by='abs(r)', ascending=False)

# FDR correction
reject, qvalue = multi.fdrcorrection(corr['p-unc'], alpha=0.05, method='indep')
corr['qvalue'] = qvalue
corr['rejected'] = reject

#### Save output

In [ ]:
corr.reset_index().to_csv(os.path.join(Base, 'tables/correlation.csv'), index=False)

In [ ]:
corr.drop('Q08380', axis=0, inplace=True)

#### Plotting

In [ ]:
# define subplot grid
fig, axs = plt.subplots(nrows=7, ncols=5, figsize=(14, 20))
plt.subplots_adjust(hspace=0.37)
plt.subplots_adjust(wspace=0.5)
#fig.suptitle("Correlation to inflammatory activity of shortlisted cytokines", fontsize=18, y=0.95)

PROPS = {
    'boxprops':{'facecolor':'none', 'edgecolor':'black'},
    'medianprops':{'color':'black'},
    'whiskerprops':{'color':'black'},
    'capprops':{'color':'black'}
}

# loop through tickers and axes
for feature, ax in zip(corr.index, axs.ravel()):
    #ax.set_xticklabels(sorted(data['inflammation'].unique().astype(int)))
    # filter df for ticker and plot on specified axes
    #df[df["ticker"] == ticker].plot(ax=ax)
    sns.boxplot(x='inflammation', y=feature, 
                data=data, ax=ax, 
                color='white', 
                width=0.6, 
                fliersize=2,
                linewidth=1, **PROPS)
    sns.stripplot(x='inflammation', y=feature, data=data,color='steelblue',ax=ax, size=2)
    r=corr.loc[feature]['r'].round(2)
    #ax.annotate(text = 'Rho={}'.format(str(r.round(2))), xy = (0.55, 0.05), xycoords = 'axes fraction', fontsize = 10)
    # chart formatting
    if feature in list(novel):
        feature_name = feature + '*'
    else:
        feature_name = feature
    ax.set_title(feature_name.upper())
    #ax.get_legend().remove()
    ax.set_xlabel('')
    ax.set_ylabel("NPX values (Log2)\n(Rho={})".format(str(r)))
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(Base, 'exploratory_analysis/correlation.pdf'), dpi=120, bbox_inches='tight')

In [ ]:
toplot = data[corr.index[:30].tolist() + ['inflammation']].groupby('inflammation').mean().T
g= sns.clustermap(toplot, col_cluster=False, z_score=0, cmap='bwr', edgecolor='white', linewidth=1)
plt.rcParams['pdf.fonttype'] = 42
g.savefig(os.path.join(Base, 'exploratory_analysis/top30.pdf'), dpi=120, bbox_inches='tight')

#### Do they also correlate with fibrosis or steatosis? 

In [ ]:
shortlist=pd.read_excel('supp_tables/ST3-5.xlsx', sheet_name='ST5')['feature'][:30].tolist()

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
to_test = list(shortlist) + ['ast_log2']
f1='steatosis'
f2='inflammation'
f3='fibrosis'
anova_results = []
for feature in to_test:
    
    formula = (
        f"{feature} ~ C({f1}) + C({f2}) + C({f3}) + "
        f"C({f2}):C({f3}) + C({f2}):C({f1})"
    )   
    
    try:
        model = ols(formula, data=data).fit()
        result = sm.stats.anova_lm(model, typ=2).copy()
        result['feature'] = feature
        anova_results.append(result)
    
    except Exception as e:
        print(f"Failed for {feature}: {e}")

In [ ]:
df_anova_results = pd.concat(anova_results)
anova_results_sig = df_anova_results[df_anova_results['PR(>F)']<0.05/35]

In [ ]:
anova_results_sig.index.value_counts()

In [ ]:
anova_results_sig.loc['C(inflammation):C(fibrosis)']

In [ ]:
anova_results_sig.loc['C(inflammation):C(steatosis)']

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))
plt.subplots_adjust(hspace=0.3)
plt.subplots_adjust(wspace=0.2)
for protein, ax in zip(['HGF', 'SCF', 'SLAMF1'], axs.ravel()):
    sns.boxplot(x='steatosis', y=protein, hue='inflammation', 
                data=data, ax=ax, palette='Blues', width=0.6, linewidth=1)
    ax.set_title(protein)
plt.rcParams['pdf.fonttype'] = 42
plt.savefig('temp.pdf')
#plt.savefig('figures/interaction.pdf', dpi=120, bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(14, 8))
plt.subplots_adjust(hspace=0.45, wspace=0.35)

proteins = ['HGF', 'SCF', 'SLAMF1', 'TWEAK', 'TRAIL']

for i, (protein, ax) in enumerate(zip(proteins, axs.ravel())):
    sns.boxplot(
        x='fibrosis', y=protein, hue='inflammation',
        data=data, ax=ax, order=['F0-1', 'F2', 'F3', 'F4'],
        palette='Blues', width=0.6, linewidth=0.8,
        flierprops=dict(marker='o', markerfacecolor='gray', markersize=3, alpha=0.5, linewidth=0)
    )
    # Title and labels
    ax.set_title(protein, fontsize=13, fontweight='bold', pad=8)
    ax.set_xlabel('Fibrosis stage', fontsize=10)
    ax.set_ylabel('Expression', fontsize=10)
    ax.tick_params(axis='both', labelsize=9)

    # Clean up spines
    sns.despine(ax=ax)

    # Legend only on first plot
    if i == 0:
        ax.legend(title='Inflammation', fontsize=8, title_fontsize=9,
                  frameon=False, loc='upper left')
    else:
        ax.get_legend().remove()

    # Panel label
    ax.text(-0.12, 1.07, list('abcde')[i] + '.', transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top')

# Hide the unused 6th panel
axs[1, 2].set_visible(False)

plt.rcParams['pdf.fonttype'] = 42
plt.savefig('temp.pdf', bbox_inches='tight', dpi=150)
#plt.savefig('figures/interaction_fibrosis.pdf', bbox_inches='tight')

### Save results

In [ ]:
proteinlist = pd.read_excel('supp_tables/olink_proteinList.xlsx')

In [ ]:
with pd.ExcelWriter('supp_tables/ST1-2-7.xlsx') as writer:
    proteinlist.to_excel(writer, sheet_name='ST1')
    corr.to_excel(writer, sheet_name='ST2')
    anova_results_sig.to_excel(writer, sheet_name='ST7')